# DocRestore — Ablation Studies

**Owner:** Sakshat  
**Week 3**

Two ablation experiments that satisfy the course requirement for 2 meaningful
model improvements backed by evidence:

1. **Loss function** — L1-only vs L1 + Perceptual (VGG16) loss
2. **Augmentation** — ShabbyPages-only vs ShabbyPages + Augraphy data

Each ablation reads the loss logs and eval CSVs produced by running training
with different configs, then visualises and interprets the results.

---

### How to run the ablation training before this notebook

```bash
# Ablation 1a — L1-only loss  (edit configs/docres.yaml: loss: l1)
python train/train_docres.py --config configs/docres_l1_only.yaml
python eval/run_eval.py --model docres --checkpoint checkpoints/docres_l1_best.pth \
    --out eval/outputs/results_docres_l1.csv

# Ablation 1b — Combined loss  (default config)
python train/train_docres.py --config configs/docres.yaml
python eval/run_eval.py --model docres --checkpoint checkpoints/docres_best.pth \
    --out eval/outputs/results_docres_combined.csv

# Ablation 2a — ShabbyPages only  (edit split: exclude augraphy data)
python train/train_docres.py --config configs/docres_shabby_only.yaml
python eval/run_eval.py --model docres --checkpoint checkpoints/docres_shabby_best.pth \
    --out eval/outputs/results_docres_shabby.csv

# Ablation 2b — ShabbyPages + Augraphy  (default, already run above)
```

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = Path("../").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

CKPT_DIR    = PROJECT_ROOT / "checkpoints"
RESULTS_DIR = PROJECT_ROOT / "eval" / "outputs"

print("Checkpoints dir :", CKPT_DIR)
print("Results dir     :", RESULTS_DIR)

---
## Ablation 1 — Loss Function: L1-only vs L1 + Perceptual

**Question:** Does adding perceptual loss (VGG16 relu2_2 features) improve
perceptual quality (SSIM) at no cost to pixel fidelity (PSNR)?

In [ ]:
# ---- Training curves ----
log_l1       = CKPT_DIR / "docres_l1_loss_log.csv"
log_combined = CKPT_DIR / "docres_loss_log.csv"

df_l1       = pd.read_csv(log_l1)       if log_l1.exists()       else None
df_combined = pd.read_csv(log_combined) if log_combined.exists() else None

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for ax, split in zip(axes, ["train_loss", "val_loss"]):
    if df_l1 is not None:
        ax.plot(df_l1["epoch"], df_l1[split],       label="L1-only",  linewidth=2)
    if df_combined is not None:
        ax.plot(df_combined["epoch"], df_combined[split], label="L1+Perceptual", linewidth=2)
    ax.set_xlabel("Epoch")
    ax.set_ylabel(split.replace("_", " ").title())
    ax.set_title(split.replace("_", " ").title())
    ax.legend()
    ax.spines[["top", "right"]].set_visible(False)

plt.suptitle("Ablation 1 — Loss Function Training Curves", fontsize=13)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "ablation1_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → eval/outputs/ablation1_curves.png")

In [ ]:
# ---- Val-set metric comparison ----
res_l1       = RESULTS_DIR / "results_docres_l1.csv"
res_combined = RESULTS_DIR / "results_docres_combined.csv"

metrics = ["psnr", "ssim", "cer"]
rows = {}

if res_l1.exists():
    df = pd.read_csv(res_l1)
    rows["L1-only"] = {m: df[m].mean() for m in metrics}

if res_combined.exists():
    df = pd.read_csv(res_combined)
    rows["L1+Perceptual"] = {m: df[m].mean() for m in metrics}

if rows:
    abl1 = pd.DataFrame(rows).T
    abl1["psnr"] = abl1["psnr"].map("{:.2f} dB".format)
    abl1["ssim"] = abl1["ssim"].map("{:.4f}".format)
    abl1["cer"]  = abl1["cer"].map("{:.2%}".format)
    print("Ablation 1 — Val-set means:")
    print(abl1.to_string())
else:
    print("[INFO] Run training and eval first to populate results CSVs.")

### Ablation 1 — Interpretation

*Fill in after results are available.*

- **Finding**: Adding perceptual loss [increases / has no effect on] SSIM by ___ while PSNR [improves by ___ dB / drops by ___ dB].
- **Conclusion**: The combined loss is [better / not worth the compute overhead] because ___.

---
## Ablation 2 — Augmentation: ShabbyPages-only vs ShabbyPages + Augraphy

**Question:** Does adding synthetically degraded Augraphy data help the model
generalise to the NoisyOffice test set (real-world scans)?

In [ ]:
# ---- Val-set metric comparison ----
res_shabby   = RESULTS_DIR / "results_docres_shabby.csv"
res_augmented = RESULTS_DIR / "results_docres_combined.csv"   # reuse from Ablation 1

rows2 = {}

if res_shabby.exists():
    df = pd.read_csv(res_shabby)
    rows2["ShabbyPages only"] = {m: df[m].mean() for m in metrics}

if res_augmented.exists():
    df = pd.read_csv(res_augmented)
    rows2["ShabbyPages + Augraphy"] = {m: df[m].mean() for m in metrics}

if rows2:
    abl2 = pd.DataFrame(rows2).T
    abl2["psnr"] = abl2["psnr"].map("{:.2f} dB".format)
    abl2["ssim"] = abl2["ssim"].map("{:.4f}".format)
    abl2["cer"]  = abl2["cer"].map("{:.2%}".format)
    print("Ablation 2 — NoisyOffice test-set means:")
    print(abl2.to_string())
else:
    print("[INFO] Run training and eval first to populate results CSVs.")

In [ ]:
# ---- Bar chart ----
if rows2:
    numeric_rows = {
        k: {m: pd.read_csv(p)[m].mean()
            for m in metrics}
        for k, p in [
            ("ShabbyPages only",     res_shabby),
            ("ShabbyPages+Augraphy", res_augmented),
        ] if Path(p).exists()
    }
    df_plot = pd.DataFrame(numeric_rows).T

    fig, axes = plt.subplots(1, 3, figsize=(13, 4))
    colors = ["#55A868", "#C44E52"]
    x = np.arange(len(df_plot))

    for ax, col, ylabel in zip(
        axes,
        ["psnr", "ssim", "cer"],
        ["PSNR (dB)  ↑", "SSIM  ↑", "CER  ↓"],
    ):
        vals = df_plot[col].values
        if col == "cer":
            vals = vals * 100
        bars = ax.bar(x, vals, color=colors[:len(x)], width=0.5)
        ax.set_title(ylabel, fontsize=12)
        ax.set_xticks(x)
        ax.set_xticklabels(df_plot.index, fontsize=9, rotation=10)
        ax.bar_label(bars, fmt="%.3f", padding=3)
        ax.spines[["top", "right"]].set_visible(False)

    plt.suptitle("Ablation 2 — Augmentation on NoisyOffice Test Set", fontsize=13)
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / "ablation2_bar.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved → eval/outputs/ablation2_bar.png")

### Ablation 2 — Interpretation

*Fill in after results are available.*

- **Finding**: Adding Augraphy-generated training pairs [improves / does not improve] PSNR on NoisyOffice by ___dB and reduces CER from ___% to ___%.
- **Conclusion**: Synthetic augmentation [does / does not] help the model generalise to real-world scan artifacts not present in ShabbyPages alone.